# Straified analyses

In [ ]:
# (1) Paths (edit here)
import os
PROJECT_ROOT = os.path.dirname(os.getcwd())
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "global40.dta")
BASE_DIR  = os.path.join(PROJECT_ROOT, "results", "f3_sub")

TAB_DIR       = os.path.join(BASE_DIR, "table")
SURV_FIGDIR   = os.path.join(BASE_DIR, "survival_pdf")
HAZ_FIGDIR    = os.path.join(BASE_DIR, "hazard_pdf")
CHZ_FIGDIR    = os.path.join(BASE_DIR, "cumhaz_pdf")
CDF_FIGDIR    = os.path.join(BASE_DIR, "cdf_pdf")

for d in [BASE_DIR, TAB_DIR, SURV_FIGDIR, HAZ_FIGDIR, CHZ_FIGDIR, CDF_FIGDIR]:
    os.makedirs(d, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("BASE_DIR :", BASE_DIR)

In [2]:
# (2) Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from scipy.optimize import minimize
import autograd.numpy as anp
from autograd import grad, hessian

print("OK: imports done.")

OK: imports done.


In [3]:
# (3) Settings
START_AGE   = 50
AGE_MAX     = 100
AGE_STEP    = 1
MIN_N       = 200
MIN_EVENTS  = 10
REPORT_AGES = [a for a in [60, 70, 80, 90, 100] if START_AGE <= a <= AGE_MAX]
DISEASE_COLS = ["rlung", "rarthr", "rstrok", "rheart", "ralzhe", "rcancr"]

W_INCH_DOUBLE = 7.0
H_INCH_DOUBLE = 4.0
W_INCH_SINGLE = 3.4
H_INCH_SINGLE = 2.4
DPI_OUT = 600
FONT_SIZE = 9

AGE_GRID = np.arange(START_AGE, AGE_MAX + 1e-9, AGE_STEP)
T_GRID   = AGE_GRID - START_AGE

print("START_AGE:", START_AGE, "AGE_MAX:", AGE_MAX, "N ages:", len(AGE_GRID))

START_AGE: 50 AGE_MAX: 100 N ages: 51


In [4]:
# (4) Utilities
def trapz(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 2 or len(y) < 2:
        return np.nan
    dx = np.diff(x)
    avg_y = (y[:-1] + y[1:]) / 2.0
    return float(np.sum(avg_y * dx))

def find_quantile_age(age: np.ndarray, surv: np.ndarray, p_surv: float = 0.5) -> float:
    if len(age) != len(surv) or np.all(pd.isna(surv)):
        return np.nan
    idx = np.where(surv <= p_surv)[0]
    if len(idx) == 0:
        return np.nan
    i = idx[0]
    if i == 0:
        return float(age[0])
    x1, x2 = age[i - 1], age[i]
    y1, y2 = surv[i - 1], surv[i]
    if np.isnan(y1) or np.isnan(y2) or y1 == y2:
        return float(age[i])
    return float(x1 + (p_surv - y1) * (x2 - x1) / (y2 - y1))

print("OK: utilities ready.")

OK: utilities ready.


In [5]:
# (5) Gompertz model (2 parameters)
# h(t)=lambda*exp(gamma*t); H(t)=(lambda/gamma)*(exp(gamma*t)-1); S(t)=exp(-H(t))
def gompertz_cumhaz(t, gamma, log_lambda):
    lambda_param = anp.exp(log_lambda)
    return (lambda_param / gamma) * anp.expm1(gamma * t)

def gompertz_hazard(t, gamma, log_lambda):
    return anp.exp(log_lambda) * anp.exp(gamma * t)

def gompertz_curves(params: np.ndarray, t_grid: np.ndarray) -> Dict[str, np.ndarray]:
    gamma, log_lambda = params[0], params[1]
    H = gompertz_cumhaz(t_grid, gamma, log_lambda)
    h = gompertz_hazard(t_grid, gamma, log_lambda)
    S = anp.exp(-H)
    return {"surv": np.asarray(S), "hazard": np.asarray(h), "cumhaz": np.asarray(H), "cdf": np.asarray(1.0 - S)}

def neg_loglik(theta, entry_t, exit_t, event):
    loglambda, loggamma = theta
    log_lambda = loglambda
    gamma = anp.exp(loggamma)

    H_exit = gompertz_cumhaz(exit_t, gamma, log_lambda)
    H_entry = gompertz_cumhaz(entry_t, gamma, log_lambda)

    logS_exit = -H_exit
    logS_entry = -H_entry
    h_exit = gompertz_hazard(exit_t, gamma, log_lambda)

    return -anp.sum(event * anp.log(h_exit + 1e-12) + logS_exit - logS_entry)

_grad_nll = grad(neg_loglik)
_hess_nll = hessian(neg_loglik)

def fit_gompertz(entry_t, exit_t, event, init=None, maxiter=2000):
    if init is None:
        init = np.array([np.log(0.01), np.log(0.01)])
    bounds = [(np.log(1e-12), np.log(1.0)), (np.log(1e-12), np.log(1.0))]
    res = minimize(
        fun=lambda th: float(neg_loglik(th, entry_t, exit_t, event)),
        x0=init,
        jac=lambda th: np.asarray(_grad_nll(th, entry_t, exit_t, event)),
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": maxiter},
    )
    theta_hat = res.x
    loglambda, loggamma = theta_hat
    log_lambda = float(loglambda)
    gamma = float(np.exp(loggamma))
    params = np.array([gamma, log_lambda])
    cov_theta = None
    try:
        hess = np.asarray(_hess_nll(theta_hat, entry_t, exit_t, event))
        cov_theta = np.linalg.inv(hess)
    except Exception:
        cov_theta = None
    cov_params = None
    if cov_theta is not None and np.all(np.isfinite(cov_theta)):
        J = np.array([[0.0, gamma], [1.0, 0.0]])
        cov_params = J @ cov_theta @ J.T
    return params, cov_params, res

def simulate_ci(params: np.ndarray, cov: np.ndarray, t_grid: np.ndarray, n_sim: int = 1000, seed: int = 1):
    if cov is None or np.any(np.isnan(cov)):
        return None
    rng = np.random.default_rng(seed)
    sim = rng.multivariate_normal(mean=params, cov=cov, size=n_sim)
    surv = np.zeros((n_sim, len(t_grid)))
    haz = np.zeros_like(surv)
    cumhaz = np.zeros_like(surv)
    cdf = np.zeros_like(surv)
    for i in range(n_sim):
        curves = gompertz_curves(sim[i], t_grid)
        surv[i] = curves["surv"]
        haz[i] = curves["hazard"]
        cumhaz[i] = curves["cumhaz"]
        cdf[i] = curves["cdf"]
    def q(x):
        return np.nanpercentile(x, [2.5, 97.5], axis=0)
    return {"surv": q(surv), "hazard": q(haz), "cumhaz": q(cumhaz), "cdf": q(cdf)}

def rmst_ci_from_params(params: np.ndarray, cov: Optional[np.ndarray], t_grid: np.ndarray, age_grid: np.ndarray, start_age: float, n_sim: int = 1000, seed: int = 1) -> Tuple[float, float]:
    if cov is None or np.any(np.isnan(cov)):
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    sim = rng.multivariate_normal(mean=params, cov=cov, size=n_sim)
    rmst_vals = np.zeros(n_sim)
    for i in range(n_sim):
        curves = gompertz_curves(sim[i], t_grid)
        surv_vals = np.clip(curves["surv"], 0, 1)
        rmst_vals[i] = start_age + trapz(age_grid, surv_vals)
    lcl, ucl = np.nanpercentile(rmst_vals, [2.5, 97.5])
    return (float(lcl), float(ucl))

print("OK: Gompertz functions ready.")

OK: Gompertz functions ready.


In [6]:
# (6) Load & clean data
raw = pd.read_stata(DATA_PATH)

def _coerce_labeled_int(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().any():
        return numeric
    return pd.to_numeric(series.astype(str).str.extract(r"^(\d+)")[0], errors="coerce")

raw = (
    raw.assign(
        id=raw["id"].astype(str),
        age=pd.to_numeric(raw["ragey_b"], errors="coerce"),
        country=raw["isocountry_c"].astype(str),
        continent=raw["continent_c"].astype(str),
        rwalk1a=pd.to_numeric(raw["rwalk1a"], errors="coerce"),
        ragender=_coerce_labeled_int(raw.get("ragender")),
        rurbrur=_coerce_labeled_int(raw.get("rurbrur")),
        rlung=pd.to_numeric(raw.get("rlung"), errors="coerce"),
        rarthr=pd.to_numeric(raw.get("rarthr"), errors="coerce"),
        rstrok=pd.to_numeric(raw.get("rstrok"), errors="coerce"),
        rheart=pd.to_numeric(raw.get("rheart"), errors="coerce"),
        ralzhe=pd.to_numeric(raw.get("ralzhe"), errors="coerce"),
        rcancr=pd.to_numeric(raw.get("rcancr"), errors="coerce"),
    )
    .loc[:, [
        "id", "age", "country", "continent", "rwalk1a", "ragender", "rurbrur",
        "rlung", "rarthr", "rstrok", "rheart", "ralzhe", "rcancr",
    ]]
    .dropna(subset=["id", "country", "age"])
    .loc[lambda d: d["country"].str.strip() != "", :]
)

raw["rwalk1a"] = raw["rwalk1a"].where(raw["rwalk1a"].isin([0, 1]))
raw["ragender"] = raw["ragender"].where(raw["ragender"].isin([1, 2]))
raw["rurbrur"] = raw["rurbrur"].where(raw["rurbrur"].isin([0, 1]))
for c in DISEASE_COLS:
    raw[c] = raw[c].where(raw[c].isin([0, 1]))

print("raw shape:", raw.shape)
print("countries:", raw["country"].nunique())

raw shape: (1206788, 13)
countries: 40


In [7]:
# (7) Build person-level data
person_rows: List[Dict] = []
for (country, pid), g in raw.sort_values(["country", "id", "age"]).groupby(["country", "id"]):
    cont = g["continent"].dropna().iloc[0] if g["continent"].notna().any() else np.nan
    valid = g.loc[g["rwalk1a"].notna(), :]
    if valid.empty:
        continue
    baseline_valid_age = valid["age"].min()
    baseline_state = valid.iloc[0]["rwalk1a"]
    censor_age = valid["age"].max()
    event_age = valid.loc[valid["rwalk1a"] == 1, "age"].min() if (valid["rwalk1a"] == 1).any() else np.nan
    if pd.isna(baseline_state) or baseline_state != 0:
        continue
    if pd.isna(baseline_valid_age) or pd.isna(censor_age) or censor_age <= baseline_valid_age:
        continue
    if censor_age <= START_AGE:
        continue
    censor_age_w = min(censor_age, AGE_MAX)
    event_w = 1 if (not pd.isna(event_age) and event_age <= AGE_MAX) else 0
    exit_age = event_age if event_w == 1 else censor_age_w
    entry_age = min(max(baseline_valid_age, START_AGE), AGE_MAX)
    exit_age = min(exit_age, AGE_MAX)
    entry_t = entry_age - START_AGE
    exit_t = exit_age - START_AGE
    if pd.isna(entry_t) or pd.isna(exit_t) or exit_t <= entry_t:
        continue
    person_rows.append(dict(
        country=str(country),
        continent=str(cont),
        id=str(pid),
        entry_age=entry_age,
        exit_age=exit_age,
        entry_t=entry_t,
        exit_t=exit_t,
        event_w=event_w,
        baseline_valid_age=baseline_valid_age,
        ragender=valid.iloc[0].get("ragender", np.nan),
        rurbrur=valid.iloc[0].get("rurbrur", np.nan),
    ))
person = pd.DataFrame(person_rows)
print("person rows:", person.shape[0])

person rows: 193731


In [8]:
# (8) Group labels
GENDER_LABELS = {1: "Male", 2: "Female"}
URBAN_LABELS = {0: "Rural", 1: "Urban"}
SOUTH_COUNTRIES = {"China", "Mexico", "Romania", "Bulgaria", "India"}

person["gender_group"] = person["ragender"].map(GENDER_LABELS)
person["urban_group"] = person["rurbrur"].map(URBAN_LABELS)
person["global_group"] = person["country"].apply(lambda c: "Global South" if str(c).strip() in SOUTH_COUNTRIES else "Global North")

print(person[["gender_group", "urban_group", "global_group"]].dropna().head())

  gender_group urban_group  global_group
0         Male       Urban  Global North
1         Male       Urban  Global North
2       Female       Urban  Global North
3         Male       Urban  Global North
5         Male       Urban  Global North


In [9]:
# (9) Fit Gompertz by group
FILL_CI = "#d4e6f1"
LINE_COL = "#2874a6"

@dataclass
class GroupResult:
    ok: bool
    group_type: str
    group: str
    pars: Optional[pd.DataFrame] = None
    metrics: Optional[pd.DataFrame] = None
    pred: Optional[pd.DataFrame] = None
    haz: Optional[pd.DataFrame] = None
    cumhaz: Optional[pd.DataFrame] = None
    cdf: Optional[pd.DataFrame] = None

def _fit_one_group(df: pd.DataFrame, group_type: str, group_name: str) -> Optional[GroupResult]:
    n = len(df)
    e = int(df["event_w"].sum())
    cens = n - e
    if n < MIN_N or e < MIN_EVENTS:
        return None
    entry_t = df["entry_t"].values
    exit_t = df["exit_t"].values
    event = df["event_w"].values.astype(int)
    params, cov, _ = fit_gompertz(entry_t, exit_t, event)
    gamma, log_lambda = params
    lambda_est = float(np.exp(log_lambda))
    curves = gompertz_curves(params, T_GRID)
    ci = simulate_ci(params, cov, T_GRID, n_sim=1000, seed=hash(group_name) % 10000)
    pred_df = pd.DataFrame({
        "group_type": group_type,
        "group": group_name,
        "age": AGE_GRID,
        "surv": curves["surv"],
        "lower": ci["surv"][0] if ci else np.nan,
        "upper": ci["surv"][1] if ci else np.nan,
    })
    haz_df = pd.DataFrame({
        "group_type": group_type,
        "group": group_name,
        "age": AGE_GRID,
        "hazard": curves["hazard"],
        "lower": ci["hazard"][0] if ci else np.nan,
        "upper": ci["hazard"][1] if ci else np.nan,
    })
    cumhaz_df = pd.DataFrame({
        "group_type": group_type,
        "group": group_name,
        "age": AGE_GRID,
        "cumhaz": curves["cumhaz"],
        "lower": ci["cumhaz"][0] if ci else np.nan,
        "upper": ci["cumhaz"][1] if ci else np.nan,
    })
    cdf_df = pd.DataFrame({
        "group_type": group_type,
        "group": group_name,
        "age": AGE_GRID,
        "cdf": curves["cdf"],
        "lower": ci["cdf"][0] if ci else np.nan,
        "upper": ci["cdf"][1] if ci else np.nan,
    })
    surv_vals = np.clip(curves["surv"], 0, 1)
    rmst_years = START_AGE + trapz(AGE_GRID, surv_vals)
    rmst_lcl, rmst_ucl = rmst_ci_from_params(params, cov, T_GRID, AGE_GRID, START_AGE, n_sim=1000, seed=hash(group_name) % 10000)
    q10_age = find_quantile_age(AGE_GRID, surv_vals, p_surv=0.10)
    q25_age = find_quantile_age(AGE_GRID, surv_vals, p_surv=0.75)
    q50_age = find_quantile_age(AGE_GRID, surv_vals, p_surv=0.50)
    q75_age = find_quantile_age(AGE_GRID, surv_vals, p_surv=0.25)
    iqr_survival = q75_age - q25_age
    high_width = q10_age - q50_age
    metrics_dict = {
        "group_type": group_type,
        "group": group_name,
        "n": n,
        "events": e,
        "rmst": rmst_years,
        "rmst_lcl": rmst_lcl,
        "rmst_ucl": rmst_ucl,
        "t50": q50_age,
        "t_at_S_0_75": q25_age,
        "t_at_S_0_25": q75_age,
        "iqr_survival": iqr_survival,
        "high_width": high_width,
    }
    for age in REPORT_AGES:
        age_idx = np.searchsorted(AGE_GRID, age)
        if age_idx < len(AGE_GRID):
            metrics_dict[f"surv_at_{age}"] = surv_vals[age_idx]
            metrics_dict[f"haz_at_{age}"] = curves["hazard"][age_idx]
    metrics_df = pd.DataFrame([metrics_dict])
    if cov is not None and np.all(np.isfinite(cov)):
        gamma_se = np.sqrt(cov[0, 0])
        lambda_se = lambda_est * np.sqrt(cov[1, 1])
    else:
        gamma_se = np.nan
        lambda_se = np.nan
    pars_df = pd.DataFrame([{
        "group_type": group_type,
        "group": group_name,
        "n": n,
        "events": e,
        "censored": cens,
        "gamma_est": float(gamma),
        "gamma_se": float(gamma_se),
        "lambda_est": float(lambda_est),
        "lambda_se": float(lambda_se),
    }])
    return GroupResult(ok=True, group_type=group_type, group=group_name, pars=pars_df, metrics=metrics_df, pred=pred_df, haz=haz_df, cumhaz=cumhaz_df, cdf=cdf_df)

def fit_groups(person_df: pd.DataFrame, group_col: str, group_type: str) -> List[GroupResult]:
    results = []
    for group_name, g in person_df.dropna(subset=[group_col]).groupby(group_col):
        res = _fit_one_group(g, group_type, str(group_name))
        if res is not None:
            results.append(res)
    return results

group_results = []
group_results += fit_groups(person, "gender_group", "gender")
group_results += fit_groups(person, "urban_group", "urban")
group_results += fit_groups(person, "global_group", "global")

print("OK: fitted groups", len(group_results))

OK: fitted groups 6


In [10]:
# (10) Save parameter & metrics tables
pars_all = pd.concat([r.pars for r in group_results if r.pars is not None], ignore_index=True)
met_all  = pd.concat([r.metrics for r in group_results if r.metrics is not None], ignore_index=True)

pars_path = os.path.join(TAB_DIR, "gompertz_params_by_group.csv")
met_path  = os.path.join(TAB_DIR, "gompertz_derived_metrics_by_group.csv")

pars_all.to_csv(pars_path, index=False)
met_all.to_csv(met_path, index=False)

print("Saved:", pars_path)
print("Saved:", met_path)

Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/table/gompertz_params_by_group.csv
Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/table/gompertz_derived_metrics_by_group.csv


In [11]:
# (11) Save curve tables
pred_all = pd.concat([r.pred for r in group_results if r.pred is not None], ignore_index=True)
haz_all = pd.concat([r.haz for r in group_results if r.haz is not None], ignore_index=True)
cumhaz_all = pd.concat([r.cumhaz for r in group_results if r.cumhaz is not None], ignore_index=True)
cdf_all = pd.concat([r.cdf for r in group_results if r.cdf is not None], ignore_index=True)

pred_all.to_csv(os.path.join(TAB_DIR, "gompertz_survival_curve_by_group.csv"), index=False)
haz_all.to_csv(os.path.join(TAB_DIR, "gompertz_hazard_curve_by_group.csv"), index=False)
cumhaz_all.to_csv(os.path.join(TAB_DIR, "gompertz_cumhaz_curve_by_group.csv"), index=False)
cdf_all.to_csv(os.path.join(TAB_DIR, "gompertz_cdf_curve_by_group.csv"), index=False)

print("Saved curve tables to:", TAB_DIR)

Saved curve tables to: /Users/karwailin/Desktop/OAwalk/results/f3_sub/table


In [12]:
# (12) Plot group curves (same group on one plot)
def theme_std(ax: plt.Axes, font_size: int = 9):
    ax.set_facecolor("none")
    ax.tick_params(axis="both", labelsize=font_size, width=0.5, length=2.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
        spine.set_color("black")
    ax.xaxis.label.set_fontsize(font_size)
    ax.yaxis.label.set_fontsize(font_size)

def save_plot(fig: plt.Figure, path: str, transparent: bool = True):
    base, _ = os.path.splitext(path)
    pdf_path = f"{base}.pdf"
    fig.patch.set_facecolor("none")
    fig.patch.set_alpha(0.0)
    fig.savefig(
        pdf_path,
        dpi=DPI_OUT,
        bbox_inches="tight",
        facecolor="none",
        edgecolor="none",
        transparent=True,
    )
    plt.close(fig)

COLOR_MAP = {
    "Female": "#FFB2AE",
    "Male": "#65B1FB", 
    "Rural": "#FFE08E",
    "Urban": "#FCA86C",
    "Global North": "#ADB2D6",
    "Global South": "#A981BD",
}

TITLE_MAP = {
    "global": "Global North vs. Global South",
    "urban": "Rural vs. Urban",
    "gender": "Female vs. Male",
}

def plot_group_curves(results: List[GroupResult], group_type: str, curve: str, y_col: str, out_dir: str, y_label: str, ylim: Optional[Tuple[float, float]] = None, title: Optional[str] = None):
    subset = [r for r in results if r.group_type == group_type]
    if not subset:
        return
    fig, ax = plt.subplots(figsize=(3.5, H_INCH_DOUBLE), dpi=600)
    for r in subset:
        df = getattr(r, curve)
        if df is None or df.empty:
            continue
        col = COLOR_MAP.get(r.group, LINE_COL)
        label = r.group
        if curve == "pred" and r.pars is not None and not r.pars.empty:
            row = r.pars.iloc[0]
            lambda_est = float(row.get("lambda_est", np.nan))
            gamma_est = float(row.get("gamma_est", np.nan))
            if np.isfinite(lambda_est) and np.isfinite(gamma_est):
                label = f"{r.group}\n($\\lambda$={lambda_est:.4f}, $\\gamma$={gamma_est:.4f})"
        ax.fill_between(df["age"], df["lower"], df["upper"], color=col, alpha=0.15, linewidth=0)
        ax.plot(df["age"], df[y_col], color=col, linewidth=1.2, label=label)
    ax.set_xlim(START_AGE, AGE_MAX)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_xlabel("Age (years)", fontsize=FONT_SIZE)
    ax.set_ylabel(y_label, fontsize=FONT_SIZE)
    if curve == "pred":
        ax.set_ylabel("")
        if group_type in {"global", "urban", "gender"}:
            ax.set_yticks([])
            ax.tick_params(axis="y", left=False, labelleft=False)
    if title:
        ax.set_title(title, fontsize=FONT_SIZE + 4)
    ax.legend(loc="upper right", frameon=False, fontsize=FONT_SIZE-1)
    theme_std(ax, font_size=FONT_SIZE)
    plt.tight_layout()
    out_path = os.path.join(out_dir, f"gompertz_{curve}_{group_type}")
    save_plot(fig, out_path)

for gt in ["gender", "urban", "global"]:
    plot_group_curves(
        group_results,
        gt,
        "pred",
        "surv",
        SURV_FIGDIR,
        "Difficulty-free probability",
        ylim=(0, 1),
        title=TITLE_MAP.get(gt),
    )
    plot_group_curves(group_results, gt, "haz", "hazard", HAZ_FIGDIR, "Hazard h(age)")
    plot_group_curves(group_results, gt, "cumhaz", "cumhaz", CHZ_FIGDIR, "Cumulative hazard H(age)")
    plot_group_curves(group_results, gt, "cdf", "cdf", CDF_FIGDIR, "CDF F(age)", ylim=(0, 1))

print("Group plots saved.")

Group plots saved.


In [13]:
# (13) CALE bar charts with t-tests
from scipy.stats import norm

CALE_FIGDIR = os.path.join(BASE_DIR, "cale_bar")
os.makedirs(CALE_FIGDIR, exist_ok=True)

def _sig_label(p: float) -> str:
    if p < 0.001:
        return "p<0.001"
    if p < 0.01:
        return "p<0.01"
    if p < 0.05:
        return "p<0.05"
    return f"p={p:.3f}"

def _rmst_se(row: pd.Series) -> float:
    if pd.isna(row.get("rmst_lcl")) or pd.isna(row.get("rmst_ucl")):
        return np.nan
    return (row["rmst_ucl"] - row["rmst_lcl"]) / (2.0 * 1.96)

def _p_value_diff(r1: pd.Series, r2: pd.Series) -> float:
    se1 = _rmst_se(r1)
    se2 = _rmst_se(r2)
    if np.isnan(se1) or np.isnan(se2):
        return np.nan
    se = np.sqrt(se1 ** 2 + se2 ** 2)
    if se <= 0:
        return np.nan
    z = (r1["rmst"] - r2["rmst"]) / se
    return float(2.0 * (1.0 - norm.cdf(abs(z))))

def _set_y_ticks(ax: plt.Axes):
    y0, y1 = ax.get_ylim()
    ticks = np.arange(65, np.floor(y1) + 1, 5)
    ticks = [t for t in ticks if int(t) != 90]
    ax.set_yticks(ticks)

def plot_cale_bar(group_type: str, order: List[str], out_name: str, display_labels: Optional[List[str]] = None):
    df = met_all.loc[met_all["group_type"] == group_type].copy()
    df = df.set_index("group").loc[order].reset_index()
    fig, ax = plt.subplots(figsize=(2, H_INCH_SINGLE), dpi=600)
    x = (np.arange(len(order)) - (len(order) - 1) / 2.0) * 0.4
    colors = [COLOR_MAP.get(g, LINE_COL) for g in order]
    bars = ax.bar(x, df["rmst"], color=colors, edgecolor="none", linewidth=0, width=0.30)
    for i, b in enumerate(bars):
        val = df.loc[i, "rmst"]
        ax.text(b.get_x() + b.get_width() / 2.0, b.get_height(), f"{val:.2f}", ha="center", va="bottom", fontsize=FONT_SIZE - 1)
    if len(order) == 2:
        p = _p_value_diff(df.loc[0], df.loc[1])
        if pd.notna(p):
            y = max(df["rmst"].max() * 1.06, 66)
            ax.plot([x[0], x[0], x[1], x[1]], [y * 0.98, y, y, y * 0.98], color="black", linewidth=0.5)
            ax.text((x[0] + x[1]) / 2.0, y * 1.01, _sig_label(p), ha="center", va="bottom", fontsize=FONT_SIZE - 1)
            ax.set_ylim(65, y * 1.04)
    ax.set_xticks(x)
    labels = display_labels if display_labels is not None else order
    ax.set_xticklabels(labels, fontsize=FONT_SIZE)
    ax.set_ylabel("CALE (years)", fontsize=FONT_SIZE)
    if ax.get_ylim()[0] < 65:
        y_top = ax.get_ylim()[1]
        ax.set_ylim(65, y_top)
    _set_y_ticks(ax)
    ax.set_facecolor("none")
    theme_std(ax, font_size=FONT_SIZE)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.3)
    ax.spines["bottom"].set_linewidth(0.3)
    plt.tight_layout()
    save_plot(fig, os.path.join(CALE_FIGDIR, out_name), transparent=True)

plot_cale_bar("gender", ["Female", "Male"], "cale_gender")
plot_cale_bar("urban", ["Rural", "Urban"], "cale_urban")
plot_cale_bar("global", ["Global North", "Global South"], "cale_global", display_labels=["GN", "GS"])

print("CALE bar charts saved.")

CALE bar charts saved.


In [14]:
# (14) CALE by country (urban residents)
urban_person = person.loc[person["urban_group"] == "Urban"].copy()

rows = []
skipped = []
for country, g in urban_person.groupby("country"):
    n = len(g)
    e = int(g["event_w"].sum())
    if n < MIN_N or e < MIN_EVENTS:
        skipped.append((country, n, e))
        continue
    params, _, _ = fit_gompertz(g["entry_t"].values, g["exit_t"].values, g["event_w"].values.astype(int))
    gamma, log_lambda = params
    lambda_est = float(np.exp(log_lambda))
    curves = gompertz_curves(params, T_GRID)
    surv_vals = np.clip(curves["surv"], 0, 1)
    rmst_years = START_AGE + trapz(AGE_GRID, surv_vals)
    rows.append({
        "country": str(country),
        "n": n,
        "events": e,
        "rmst": float(rmst_years),
        "gamma_est": float(gamma),
        "lambda_est": float(lambda_est),
    })

urban_cale = pd.DataFrame(rows).sort_values("country").reset_index(drop=True)
out_path = os.path.join(TAB_DIR, "gompertz_derived_metrics_by_country_urban.csv")
urban_cale.to_csv(out_path, index=False)
print("Saved:", out_path)
print("Countries fitted:", len(urban_cale))
print("Countries skipped (n/events):", len(skipped))
urban_cale.head(40)

Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/table/gompertz_derived_metrics_by_country_urban.csv
Countries fitted: 35
Countries skipped (n/events): 0


,country,n,events,rmst,gamma_est,lambda_est
0,Australia,1427,749,78.024900,0.085530,0.005367
1,Austria,2968,534,82.440434,0.089984,0.003075
2,Belgium,5041,866,81.594151,0.087256,0.003584
3,Bulgaria,459,48,78.373288,0.098832,0.003891
4,China,7397,691,78.871056,0.096230,0.003898
5,Croatia,1755,257,76.737258,0.094932,0.005063
6,Cyprus,468,71,78.205398,0.099072,0.003944
7,Czech Republic,3724,676,80.387589,0.083143,0.004454
8,Denmark,3395,452,84.309762,0.092524,0.002362
9,Estonia,3908,891,78.973146,0.076597,0.005905


In [15]:

# (15) CALE, gamma, and lambda by country for urban-rural and female-male groups
import zlib

COUNTRY_GROUP_DIR = os.path.join(BASE_DIR, "country_group_pdf")
os.makedirs(COUNTRY_GROUP_DIR, exist_ok=True)

REF_COUNTRY_PATH = os.path.join(PROJECT_ROOT, "results", "f2_gompertz", "table", "gompertz_derived_metrics_by_country.csv")
if os.path.exists(REF_COUNTRY_PATH):
    COUNTRY_PANEL_ORDER = pd.read_csv(REF_COUNTRY_PATH)["country"].astype(str).str.strip().drop_duplicates().tolist()
else:
    COUNTRY_PANEL_ORDER = sorted(person["country"].dropna().astype(str).str.strip().unique().tolist())[:36]

COUNTRY_GROUP_SPECS = {
    "urban": {"column": "urban_group", "groups": ["Rural", "Urban"]},
    "gender": {"column": "gender_group", "groups": ["Female", "Male"]},
}

def _stable_seed(*parts) -> int:
    key = "|".join(str(p) for p in parts)
    return zlib.crc32(key.encode("utf-8")) & 0xffffffff

def _fit_country_group(country: str, group_type: str, group_name: str, df: pd.DataFrame) -> Optional[Dict[str, pd.DataFrame]]:
    n = len(df)
    e = int(df["event_w"].sum())
    cens = n - e
    if n < MIN_N or e < MIN_EVENTS:
        return None
    entry_t = df["entry_t"].values
    exit_t = df["exit_t"].values
    event = df["event_w"].values.astype(int)
    params, cov, _ = fit_gompertz(entry_t, exit_t, event)
    gamma, log_lambda = params
    lambda_est = float(np.exp(log_lambda))
    curves = gompertz_curves(params, T_GRID)
    ci = simulate_ci(params, cov, T_GRID, n_sim=500, seed=_stable_seed(country, group_type, group_name))
    surv_vals = np.clip(curves["surv"], 0, 1)
    rmst_years = START_AGE + trapz(AGE_GRID, surv_vals)
    rmst_lcl, rmst_ucl = rmst_ci_from_params(
        params, cov, T_GRID, AGE_GRID, START_AGE, n_sim=500,
        seed=_stable_seed(country, group_type, group_name, "rmst")
    )
    if cov is not None and np.all(np.isfinite(cov)):
        gamma_se = np.sqrt(cov[0, 0])
        lambda_se = lambda_est * np.sqrt(cov[1, 1])
    else:
        gamma_se = np.nan
        lambda_se = np.nan
    continent = df["continent"].dropna().iloc[0] if df["continent"].notna().any() else np.nan
    metric_row = {
        "country": str(country),
        "continent": str(continent),
        "group_type": group_type,
        "group": group_name,
        "n": n,
        "events": e,
        "censored": cens,
        "rmst": float(rmst_years),
        "rmst_lcl": rmst_lcl,
        "rmst_ucl": rmst_ucl,
        "gamma_est": float(gamma),
        "gamma_se": float(gamma_se),
        "lambda_est": lambda_est,
        "lambda_se": float(lambda_se),
    }
    curve_df = pd.DataFrame({
        "country": str(country),
        "continent": str(continent),
        "group_type": group_type,
        "group": group_name,
        "age": AGE_GRID,
        "surv": curves["surv"],
        "lower": ci["surv"][0] if ci else np.nan,
        "upper": ci["surv"][1] if ci else np.nan,
        "gamma_est": float(gamma),
        "lambda_est": lambda_est,
    })
    return {"metrics": pd.DataFrame([metric_row]), "survival": curve_df}

country_group_metrics = []
country_group_survival = []
country_group_skipped = []
for group_type, spec in COUNTRY_GROUP_SPECS.items():
    group_col = spec["column"]
    for country in COUNTRY_PANEL_ORDER:
        cdf = person.loc[person["country"].astype(str).str.strip() == country].copy()
        for group_name in spec["groups"]:
            gdf = cdf.loc[cdf[group_col] == group_name].copy()
            fit = _fit_country_group(country, group_type, group_name, gdf)
            if fit is None:
                country_group_skipped.append({
                    "country": country,
                    "group_type": group_type,
                    "group": group_name,
                    "n": len(gdf),
                    "events": int(gdf["event_w"].sum()) if len(gdf) else 0,
                })
                continue
            country_group_metrics.append(fit["metrics"])
            country_group_survival.append(fit["survival"])

country_group_metrics = pd.concat(country_group_metrics, ignore_index=True) if country_group_metrics else pd.DataFrame()
country_group_survival = pd.concat(country_group_survival, ignore_index=True) if country_group_survival else pd.DataFrame()
country_group_skipped = pd.DataFrame(country_group_skipped)

metrics_path = os.path.join(TAB_DIR, "gompertz_metrics_by_country_urban_gender.csv")
surv_path = os.path.join(TAB_DIR, "gompertz_survival_by_country_urban_gender.csv")
skip_path = os.path.join(TAB_DIR, "gompertz_skipped_by_country_urban_gender.csv")
country_group_metrics.to_csv(metrics_path, index=False)
country_group_survival.to_csv(surv_path, index=False)
country_group_skipped.to_csv(skip_path, index=False)

print("Saved:", metrics_path)
print("Saved:", surv_path)
print("Saved:", skip_path)
print("Countries in panel order:", len(COUNTRY_PANEL_ORDER))
print("Successful country-group fits:", len(country_group_metrics))
print("Skipped country-group fits:", len(country_group_skipped))
country_group_metrics.head()


Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/table/gompertz_metrics_by_country_urban_gender.csv
Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/table/gompertz_survival_by_country_urban_gender.csv
Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/table/gompertz_skipped_by_country_urban_gender.csv
Countries in panel order: 36
Successful country-group fits: 140
Skipped country-group fits: 4


,country,continent,group_type,group,n,events,censored,rmst,rmst_lcl,rmst_ucl,gamma_est,gamma_se,lambda_est,lambda_se
0,Australia,Oceania,urban,Urban,1427,749,678,78.024900,76.253542,79.379615,0.085530,0.005891,0.005367,0.001071
1,Austria,Europe,urban,Rural,1695,263,1432,82.345108,81.284012,83.265904,0.119729,0.007149,0.001483,0.000304
2,Austria,Europe,urban,Urban,2968,534,2434,82.440434,81.655186,83.169983,0.089984,0.004748,0.003075,0.000402
3,Belgium,Europe,urban,Rural,1713,320,1393,79.263208,78.162298,80.419215,0.076653,0.005332,0.005736,0.000775
4,Belgium,Europe,urban,Urban,5041,866,4175,81.594151,80.821951,82.286384,0.087256,0.003381,0.003584,0.000327


In [16]:
# (16) 6x6 country panels for urban-rural and female-male survival curves
COUNTRY_GROUP_PANEL_DIR = os.path.join(SURV_FIGDIR, "country_group_panels")
os.makedirs(COUNTRY_GROUP_PANEL_DIR, exist_ok=True)

PANEL_GROUPS = {
    "urban": ["Rural", "Urban"],
    "gender": ["Female", "Male"],
}

def _format_country_group_axis(ax):
    ax.set_xlim(START_AGE, AGE_MAX)
    ax.set_ylim(0, 1)
    ax.set_xticks([START_AGE, AGE_MAX])
    ax.set_yticks([0, 0.5, 1])
    ax.set_yticklabels(["0", "50", "100"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

if "country_group_survival" not in globals() or country_group_survival.empty:
    country_group_survival = pd.read_csv(os.path.join(TAB_DIR, "gompertz_survival_by_country_urban_gender.csv"))
if "country_group_metrics" not in globals() or country_group_metrics.empty:
    country_group_metrics = pd.read_csv(os.path.join(TAB_DIR, "gompertz_metrics_by_country_urban_gender.csv"))

def _complete_country_order_for_group(group_type: str, groups: List[str]) -> List[str]:
    fit_counts = (
        country_group_metrics.loc[country_group_metrics["group_type"].eq(group_type)]
        .groupby("country")["group"]
        .apply(lambda s: set(s.dropna().astype(str)))
    )
    return [c for c in COUNTRY_PANEL_ORDER if set(groups).issubset(fit_counts.get(c, set()))]

def plot_country_group_survival_panel(group_type: str):
    groups = PANEL_GROUPS[group_type]
    countries = _complete_country_order_for_group(group_type, groups)[:36]
    nrows, ncols = 6, 6
    fig, axes = plt.subplots(nrows, ncols, figsize=(14.0, 14.0), dpi=DPI_OUT, sharex=True, sharey=True)
    axes_flat = axes.ravel()
    for ax_idx, ax in enumerate(axes_flat):
        if ax_idx >= len(countries):
            ax.set_visible(False)
            continue
        country = countries[ax_idx]
        cdf = country_group_survival.loc[
            (country_group_survival["group_type"] == group_type) &
            (country_group_survival["country"].astype(str).str.strip() == country)
        ].copy()
        mdf = country_group_metrics.loc[
            (country_group_metrics["group_type"] == group_type) &
            (country_group_metrics["country"].astype(str).str.strip() == country)
        ].copy()
        param_lines = []
        for group_name in groups:
            gcurve = cdf.loc[cdf["group"] == group_name].sort_values("age")
            if gcurve.empty:
                continue
            color = COLOR_MAP.get(group_name, LINE_COL)
            if {"lower", "upper"}.issubset(gcurve.columns):
                ax.fill_between(gcurve["age"], gcurve["lower"], gcurve["upper"], color=color, alpha=0.12, linewidth=0, zorder=1)
            ax.plot(gcurve["age"], gcurve["surv"], color=color, linewidth=1.0, zorder=3)
            grow = mdf.loc[mdf["group"] == group_name]
            if not grow.empty:
                lambda_est = pd.to_numeric(grow["lambda_est"].iloc[0], errors="coerce")
                gamma_est = pd.to_numeric(grow["gamma_est"].iloc[0], errors="coerce")
                if np.isfinite(lambda_est) and np.isfinite(gamma_est):
                    param_lines.extend([
                        f"{group_name}: $\\lambda$={lambda_est:.4f}",
                        f"{group_name}: $\\gamma$={gamma_est:.4f}",
                    ])
        if param_lines:
            ax.text(
                0.04,
                0.06,
                "\n".join(param_lines),
                transform=ax.transAxes,
                ha="left",
                va="bottom",
                fontsize=6.8,
                linespacing=1.12,
                color="#222222",
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.65, "pad": 1.1},
                zorder=8,
            )
        _format_country_group_axis(ax)
        ax.add_patch(plt.Rectangle((0, 1.01), 1, 0.13, transform=ax.transAxes, facecolor="#F1F3F5", edgecolor="none", clip_on=False, zorder=0))
        ax.text(0.5, 1.075, country, transform=ax.transAxes, ha="center", va="center", fontsize=10, color="black", zorder=5)
        ax.tick_params(axis="both", labelsize=9, length=2)
        if ax_idx % ncols == 0:
            ax.set_ylabel("Difficulty-free (%)", fontsize=10)
        if ax_idx // ncols == nrows - 1:
            ax.set_xlabel("Age (years)", fontsize=10)
    fig.tight_layout(rect=[0, 0.01, 1, 0.985], h_pad=1.35, w_pad=0.65)
    out_path = os.path.join(COUNTRY_GROUP_PANEL_DIR, f"survival_{group_type}_country_6x6.pdf")
    fig.savefig(out_path, dpi=DPI_OUT, bbox_inches="tight", facecolor="white", edgecolor="none")
    plt.close(fig)
    print("Saved:", out_path)
    print(f"Countries plotted for {group_type}:", len(countries))

plot_country_group_survival_panel("urban")
plot_country_group_survival_panel("gender")


Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/survival_pdf/country_group_panels/survival_urban_country_6x6.pdf
Countries plotted for urban: 33


Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/survival_pdf/country_group_panels/survival_gender_country_6x6.pdf
Countries plotted for gender: 36


In [17]:
# (17) CALE lollipop by group: one PDF for urban-rural, one PDF for female-male
from matplotlib.ticker import FuncFormatter

LAST_SURVEY_YEAR_PATH = os.path.join(PROJECT_ROOT, "results", "model_prep", "last_survey_year_by_country.csv")

if "country_group_metrics" not in globals() or country_group_metrics.empty:
    country_group_metrics = pd.read_csv(os.path.join(TAB_DIR, "gompertz_metrics_by_country_urban_gender.csv"))

if "COUNTRY_PANEL_ORDER" not in globals():
    REF_COUNTRY_PATH = os.path.join(PROJECT_ROOT, "results", "f2_gompertz", "table", "gompertz_derived_metrics_by_country.csv")
    if os.path.exists(REF_COUNTRY_PATH):
        COUNTRY_PANEL_ORDER = pd.read_csv(REF_COUNTRY_PATH)["country"].astype(str).str.strip().drop_duplicates().tolist()
    else:
        COUNTRY_PANEL_ORDER = sorted(country_group_metrics["country"].dropna().astype(str).str.strip().unique().tolist())[:36]

LOLLIPOP_DIR = os.path.join(BASE_DIR, "cale_lollipop")
os.makedirs(LOLLIPOP_DIR, exist_ok=True)

BASE_AGE = 65
LABEL_OFFSET = 1.05
LINE_WIDTH = 1.8
POINT_SIZE = 24

def _last_survey_country_table(countries: List[str]) -> pd.DataFrame:
    order_df = pd.DataFrame({"country": [str(c).strip() for c in countries]})
    if os.path.exists(LAST_SURVEY_YEAR_PATH):
        last_year = pd.read_csv(LAST_SURVEY_YEAR_PATH)
        last_year["country"] = last_year["country"].astype(str).str.strip()
        last_year["last_survey_year"] = pd.to_numeric(last_year["last_survey_year"], errors="coerce")
        order_df = order_df.merge(last_year[["country", "last_survey_year"]], on="country", how="left")
    else:
        order_df["last_survey_year"] = np.nan
    order_df = order_df.sort_values(["last_survey_year", "country"], ascending=[False, True], na_position="last")
    order_df["display_name"] = order_df["country"]
    year_mask = order_df["last_survey_year"].notna()
    order_df.loc[year_mask, "display_name"] = (
        order_df.loc[year_mask, "country"]
        + " ("
        + order_df.loc[year_mask, "last_survey_year"].astype(int).astype(str)
        + ")"
    )
    return order_df.reset_index(drop=True)

LOLLIPOP_COUNTRY_TABLE = _last_survey_country_table(COUNTRY_PANEL_ORDER[:36])
LOLLIPOP_COUNTRY_ORDER = LOLLIPOP_COUNTRY_TABLE["country"].tolist()

def _plot_cale_lollipop_pair(group_type: str, left_group: str, right_group: str, out_name: str):
    df = country_group_metrics.loc[
        country_group_metrics["group_type"].eq(group_type),
        ["country", "group", "rmst"],
    ].copy()
    wide_all = df.pivot_table(index="country", columns="group", values="rmst", aggfunc="first")
    complete_order = [
        c for c in LOLLIPOP_COUNTRY_ORDER
        if c in wide_all.index and pd.notna(wide_all.at[c, left_group]) and pd.notna(wide_all.at[c, right_group])
    ]
    wide = wide_all.reindex(complete_order).reset_index()
    names = wide["country"].astype(str).tolist()
    display_names = (
        pd.DataFrame({"country": names})
        .merge(LOLLIPOP_COUNTRY_TABLE[["country", "display_name"]], on="country", how="left")
        ["display_name"]
        .fillna(pd.Series(names))
        .tolist()
    )
    y_pos = np.arange(len(names))
    vals_all = pd.to_numeric(wide[[left_group, right_group]].stack(), errors="coerce")
    cale_max = float(vals_all.max()) if vals_all.notna().any() else 90
    x_max = max(90, np.ceil(cale_max + 2))
    y_ticks = [65, 75, 85] if x_max <= 92 else [65, 75, 85, 95]

    fig_h = max(6.5, 0.26 * len(names))
    fig = plt.figure(figsize=(6.8, fig_h), dpi=DPI_OUT)
    fig.patch.set_facecolor("none")
    fig.patch.set_alpha(0.0)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.05, 0.85, 1.05], wspace=0.0)
    ax_left = fig.add_subplot(gs[0, 0])
    ax_mid = fig.add_subplot(gs[0, 1], sharey=ax_left)
    ax_right = fig.add_subplot(gs[0, 2], sharey=ax_left)
    for ax in [ax_left, ax_mid, ax_right]:
        ax.set_facecolor("none")

    left_vals = pd.to_numeric(wide[left_group], errors="coerce").to_numpy(dtype=float)
    right_vals = pd.to_numeric(wide[right_group], errors="coerce").to_numpy(dtype=float)
    left_mask = np.isfinite(left_vals)
    right_mask = np.isfinite(right_vals)
    left_color = COLOR_MAP.get(left_group, LINE_COL)
    right_color = COLOR_MAP.get(right_group, LINE_COL)

    ax_left.hlines(y=y_pos[left_mask], xmin=0, xmax=-(left_vals[left_mask] - BASE_AGE), color=left_color, alpha=0.75, linewidth=LINE_WIDTH)
    ax_left.scatter(-(left_vals[left_mask] - BASE_AGE), y_pos[left_mask], color=left_color, s=POINT_SIZE, zorder=3)
    for idx in np.where(left_mask)[0]:
        ax_left.text(-(left_vals[idx] - BASE_AGE) - LABEL_OFFSET, y_pos[idx], f"{left_vals[idx]:.2f}", ha="right", va="center", fontsize=FONT_SIZE - 2, color="#333333")

    ax_mid.set_xlim(-1.8, 1.8)
    ax_mid.set_xticks([])
    ax_mid.set_yticks([])
    ax_mid.spines[:].set_visible(False)
    for y, name in zip(y_pos, display_names):
        ax_mid.text(0, y, name, ha="center", va="center", fontsize=FONT_SIZE)

    ax_right.hlines(y=y_pos[right_mask], xmin=BASE_AGE, xmax=right_vals[right_mask], color=right_color, alpha=0.75, linewidth=LINE_WIDTH)
    ax_right.scatter(right_vals[right_mask], y_pos[right_mask], color=right_color, s=POINT_SIZE, zorder=3)
    for idx in np.where(right_mask)[0]:
        ax_right.text(right_vals[idx] + LABEL_OFFSET, y_pos[idx], f"{right_vals[idx]:.2f}", ha="left", va="center", fontsize=FONT_SIZE - 2, color="#333333")

    ax_left.set_ylim(len(names) - 0.4, -0.6)
    ax_left.set_xlim(-(x_max - BASE_AGE), 0)
    ax_left.set_xticks([-(t - BASE_AGE) for t in y_ticks])
    ax_left.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{BASE_AGE + abs(x):g}"))
    ax_left.tick_params(axis="y", labelleft=False)
    ax_left.tick_params(axis="x", labelsize=FONT_SIZE - 1)
    ax_left.spines["top"].set_visible(False)
    ax_left.spines["right"].set_visible(False)
    ax_left.spines["left"].set_visible(False)
    ax_left.set_xlabel(f"CALE ({left_group})", fontsize=FONT_SIZE)

    ax_right.set_ylim(len(names) - 0.4, -0.6)
    ax_right.set_xlim(BASE_AGE, x_max)
    ax_right.set_xticks(y_ticks)
    ax_right.tick_params(axis="y", labelleft=False)
    ax_right.tick_params(axis="x", labelsize=FONT_SIZE - 1)
    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)
    ax_right.spines["left"].set_visible(False)
    ax_right.set_xlabel(f"CALE ({right_group})", fontsize=FONT_SIZE)

    plt.tight_layout()
    out_path = os.path.join(LOLLIPOP_DIR, out_name)
    fig.savefig(out_path, dpi=DPI_OUT, bbox_inches="tight", pad_inches=0, facecolor="none", edgecolor="none", transparent=True)
    plt.close(fig)
    print("Saved:", out_path)

_plot_cale_lollipop_pair("urban", "Rural", "Urban", "cale_lollipop_rural_urban.pdf")
_plot_cale_lollipop_pair("gender", "Female", "Male", "cale_lollipop_female_male.pdf")


Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/cale_lollipop/cale_lollipop_rural_urban.pdf


Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/cale_lollipop/cale_lollipop_female_male.pdf


In [18]:
# (18) Radial bar plots: gender-specific CALE - LE gap by country
from matplotlib import colors as mcolors

GENDER_LE_PATH = os.path.join(PROJECT_ROOT, "data", "wb", "wb_life_expectancy_last_survey_year.csv")
GENDER_RADIAL_DIR = os.path.join(BASE_DIR, "cale_le_gap_radial")
os.makedirs(GENDER_RADIAL_DIR, exist_ok=True)

if "country_group_metrics" not in globals() or country_group_metrics.empty:
    country_group_metrics = pd.read_csv(os.path.join(TAB_DIR, "gompertz_metrics_by_country_urban_gender.csv"))

wb_gender_le = pd.read_csv(GENDER_LE_PATH)
wb_gender_le["country"] = wb_gender_le["country"].astype(str).str.strip()
for col in ["female", "male", "last_survey_year"]:
    wb_gender_le[col] = pd.to_numeric(wb_gender_le[col], errors="coerce")


def _polar_label(ax, angle, radius, text, color="black", fs=6, fw="normal"):
    angle_deg = np.degrees(angle)
    screen_angle = 90 - angle_deg
    rotation = screen_angle - 90
    rotation = (rotation + 360) % 360
    if rotation > 180:
        rotation -= 360
    if not (-90 <= rotation <= 90):
        rotation += 180
    ax.text(
        angle,
        radius,
        text,
        rotation=rotation,
        rotation_mode="anchor",
        ha="center",
        va="center",
        fontsize=fs,
        fontweight=fw,
        color=color,
    )


def _gap_colors(values: np.ndarray) -> np.ndarray:
    neg_hex = ["#427CC4", "#528EC6", "#65A0CF", "#EEF9FF"]
    pos_hex = ["#FFEFEB", "#FFD3C5", "#F18B7C", "#F06054"]
    neg_cmap = mcolors.LinearSegmentedColormap.from_list("custom_neg", neg_hex)
    pos_cmap = mcolors.LinearSegmentedColormap.from_list("custom_pos", pos_hex)
    colors = np.zeros((len(values), 4))
    neg_mask = values < 0
    pos_mask = values > 0
    if np.any(neg_mask):
        neg_norm = mcolors.Normalize(vmin=values[neg_mask].min(), vmax=values[neg_mask].max())
        colors[neg_mask] = neg_cmap(neg_norm(values[neg_mask]))
    if np.any(pos_mask):
        pos_norm = mcolors.Normalize(vmin=values[pos_mask].min(), vmax=values[pos_mask].max())
        colors[pos_mask] = pos_cmap(pos_norm(values[pos_mask]))
    colors[~(neg_mask | pos_mask)] = mcolors.to_rgba("#EEF9FF")
    return colors


def plot_gender_cale_le_gap_radial(group_name: str, le_col: str, out_name: str):
    cale = country_group_metrics.loc[
        (country_group_metrics["group_type"] == "gender") &
        (country_group_metrics["group"] == group_name),
        ["country", "rmst"],
    ].copy()
    cale = cale.rename(columns={"rmst": "CALE"})
    cale["country"] = cale["country"].astype(str).str.strip()
    merged = cale.merge(
        wb_gender_le[["country", "iso3c", "last_survey_year", le_col]],
        on="country",
        how="inner",
    )
    merged = merged.rename(columns={le_col: "LE"})
    merged = merged.dropna(subset=["CALE", "LE"])
    merged["DIFF"] = merged["CALE"] - merged["LE"]
    merged = merged.sort_values("DIFF").reset_index(drop=True)
    merged["ISO3"] = merged["iso3c"].fillna(merged["country"]).astype(str).str.upper().str[:3]

    labels = merged["ISO3"].tolist()
    values = merged["DIFF"].to_numpy(dtype=float)
    inner_radius = 20
    theta = np.linspace(0.0, 2 * np.pi, len(values), endpoint=False)
    width = 2 * np.pi / len(values)
    widths = np.full(len(values), width * 0.9)
    colors = _gap_colors(values)

    fig = plt.figure(figsize=(7.2, 7.2), dpi=DPI_OUT)
    fig.patch.set_facecolor("none")
    fig.patch.set_alpha(0.0)
    ax = fig.add_subplot(111, projection="polar")
    ax.set_facecolor("none")
    ax.set_theta_direction(-1)
    ax.set_theta_offset(0.0)
    ax.bar(theta, values, width=widths, bottom=inner_radius, color=colors, alpha=0.9)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.grid(False)
    ax.spines["polar"].set_visible(False)

    max_abs = np.nanmax(np.abs(values)) if len(values) else 1
    ax.set_ylim(0, inner_radius + max_abs + 1.2)
    theta_dense = np.linspace(0.0, 2 * np.pi, 360)
    ax.plot(theta_dense, np.full_like(theta_dense, inner_radius), color="#BEC3C8", linewidth=0.8)

    for ang, name, v in zip(theta, labels, values):
        label_r = inner_radius - 2 if v >= 0 else inner_radius + 2
        _polar_label(ax, ang, label_r, name, color="#000000", fs=FONT_SIZE - 1)

    fig_w, fig_h = fig.get_size_inches()
    bbox = ax.get_position()
    ax_w = fig_w * bbox.width
    ax_h = fig_h * bbox.height
    radius_in = min(ax_w, ax_h) / 2.0
    cm_in = 0.35 / 2.54
    r_range = ax.get_rmax() - ax.get_rmin()
    offset_r = cm_in / radius_in * r_range
    for ang, v in zip(theta, values):
        tip_r = inner_radius + v
        offset = offset_r if v >= 0 else -offset_r
        ax.text(ang, tip_r + offset, f"{v:.2f}", fontsize=FONT_SIZE - 2, ha="center", va="center")

    out_path = os.path.join(GENDER_RADIAL_DIR, out_name)
    plt.tight_layout()
    fig.savefig(out_path, dpi=DPI_OUT, bbox_inches="tight", pad_inches=0, facecolor="none", edgecolor="none", transparent=True)
    plt.close(fig)
    print("Saved:", out_path)
    print(f"Countries matched for {group_name}:", len(merged))

plot_gender_cale_le_gap_radial("Female", "female", "cale_le_gap_radial_female.pdf")
plot_gender_cale_le_gap_radial("Male", "male", "cale_le_gap_radial_male.pdf")


Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/cale_le_gap_radial/cale_le_gap_radial_female.pdf
Countries matched for Female: 36


Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/cale_le_gap_radial/cale_le_gap_radial_male.pdf
Countries matched for Male: 36


In [19]:
# (19) Significance tests for Gompertz parameters between paired groups
from scipy.stats import norm

PARAM_TEST_PATH = os.path.join(TAB_DIR, "gompertz_parameter_pairwise_tests_by_group.csv")

if "pars_all" not in globals() or pars_all.empty:
    pars_all = pd.read_csv(os.path.join(TAB_DIR, "gompertz_params_by_group.csv"))

PAIRWISE_PARAM_TESTS = {
    "gender": ("Female", "Male"),
    "urban": ("Rural", "Urban"),
    "global": ("Global North", "Global South"),
}


def _p_from_independent_estimates(est1: float, se1: float, est2: float, se2: float) -> float:
    if not all(np.isfinite([est1, se1, est2, se2])):
        return np.nan
    se_diff = np.sqrt(se1 ** 2 + se2 ** 2)
    if se_diff <= 0:
        return np.nan
    z = (est1 - est2) / se_diff
    return float(2.0 * (1.0 - norm.cdf(abs(z))))


def _fmt_p_value(p: float) -> str:
    if pd.isna(p):
        return "p=NA"
    if p < 0.001:
        return "p<0.001"
    return f"p={p:.3f}"



rows = []
for group_type, (g1, g2) in PAIRWISE_PARAM_TESTS.items():
    sub = pars_all.loc[pars_all["group_type"].eq(group_type)].copy()
    r1 = sub.loc[sub["group"].eq(g1)]
    r2 = sub.loc[sub["group"].eq(g2)]
    if r1.empty or r2.empty:
        print(f"{group_type}: missing {g1} or {g2}; skipped")
        continue
    r1 = r1.iloc[0]
    r2 = r2.iloc[0]
    for param in ["gamma", "lambda"]:
        est_col = f"{param}_est"
        se_col = f"{param}_se"
        p_value = _p_from_independent_estimates(
            float(r1[est_col]), float(r1[se_col]),
            float(r2[est_col]), float(r2[se_col]),
        )
        rows.append({
            "group_type": group_type,
            "group_1": g1,
            "group_2": g2,
            "parameter": param,
            "estimate_1": float(r1[est_col]),
            "se_1": float(r1[se_col]),
            "estimate_2": float(r2[est_col]),
            "se_2": float(r2[se_col]),
            "p_value": p_value,
        })
        print(
            f"{group_type}: {g1} vs {g2}, {param}: "
            f"{r1[est_col]:.6g} vs {r2[est_col]:.6g}, {_fmt_p_value(p_value)}"
        )

param_pairwise_tests = pd.DataFrame(rows)
param_pairwise_tests.to_csv(PARAM_TEST_PATH, index=False)
print("Saved:", PARAM_TEST_PATH)


gender: Female vs Male, gamma: 0.0773849 vs 0.0816964, p<0.001
gender: Female vs Male, lambda: 0.00576619 vs 0.00421212, p<0.001
urban: Rural vs Urban, gamma: 0.0720372 vs 0.0827719, p<0.001
urban: Rural vs Urban, lambda: 0.00682272 vs 0.00431313, p<0.001
global: Global North vs Global South, gamma: 0.0804531 vs 0.0802704, p=0.894
global: Global North vs Global South, lambda: 0.00471726 vs 0.00613737, p<0.001
Saved: /Users/karwailin/Desktop/OAwalk/results/f3_sub/table/gompertz_parameter_pairwise_tests_by_group.csv
